<a href="https://colab.research.google.com/github/brandonviaje/Clarity/blob/main/Clarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clarity: Semantic Vision Engine**

**Group ID:** 14

**Course:** CSCI4052U  

**Brandon Viaje (100912514)**


## Overview

<img src="https://www.displayr.com/wp-content/uploads/2024/10/3-tips-sentiment-analysis-1024x625.png" width="500" height="auto">

A picture is worth a thousand words, but traditional computer vision models often fail to capture the full **semantic meaning** of an image. **Clarity** is a multi-model neural pipeline designed to bridge the gap between **raw pixels and human-like understanding.**

In addition, while traditional systems are limited to **closed-set object detection** (recognizing only what they were trained on), Clarity leverages **Open-Vocabulary Perception** to reason about novel objects, ambiguous scenes, and complex visual contexts.

### **The Core Mission**
- **Generate** semantic descriptions of raw visual data.
- **Extract** meaningful object-level concepts using NLP.
- **Execute** open-vocabulary detection and scene classification.

---

## How It Works

Clarity operates as a multi-stage inference system where each model's output informs the next:

1.  **Input Stage:** User uploads a raw RGB image.
2.  **Caption Generation (BLIP):** The image is transformed into a descriptive sentence.
3.  **NLP Processing (spaCy):** The system parses the caption to extract key nouns and entities.
4.  **Object Detection (YOLO-World):** These extracted entities become dynamic detection targets for bounding box regression.
5.  **Scene Classification (CLIP):** The image is matched against semantic scene prompts to provide global context.
6.  **Output:** A final output containing the caption, detected objects, and scene labels.

---

## Data Flow

**Raw Image**  
   ↓  
**BLIP (Captioning)**  
   ↓  
**spaCy (Entity Extraction)**  
   ↓  
**YOLO-World (Dynamic Detection)**  
   ↓  
**CLIP (Global Classification)**  
   ↓  
**Structured Scene Understanding**

---

## Applications
* **Autonomous Systems:** Navigation in previously unseen environments.
* **Assistive Tech:** Real-time visual description for the visually impaired.
* **Advanced Monitoring:** Identifying specific behaviors or objects without pre-defined labels.

# Install Libraries and Dependencies

In [1]:
# install libraries
!pip install torch torchvision transformers ultralytics spacy gradio Pillow timm
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 105.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Neural Network Components: How They Work Together

This system is a **multi-model neural pipeline** composed of pretrained vision-language and object detection models. Rather than training models from scratch, we perform **zero-shot inference**, leveraging pretrained knowledge from large-scale datasets.

Each component plays a specialized role in the overall perception pipeline.

## Neural Network Architecture

The engine integrates three state-of-the-art pretrained architectures into a unified reasoning pipeline:

| Feature | BLIP | YOLO-World | CLIP |
| :-- | :-- | :-- | :-- |
| Purpose | Image Captioning | Object Detection | Scene Classification |
| Architecture | Vision Transformer | CNN + Text Embedding | Dual Transformer Encoder |
| Output | Text | Bounding Boxes | Probability Scores |
| Role in System | Semantic Description | Local Object Localization | Global Context Understanding |


## Overview

Together, these models form a **vision understanding system**:

- BLIP provides **semantic interpretation**
- YOLO-World provides **spatial object grounding**
- CLIP provides **global scene reasoning**

This combination enables robust **open-vocabulary environmental perception** without task-specific retraining.

In [2]:
# import libraries to load our models
import torch
from PIL import Image
from ultralytics import YOLO
from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    CLIPProcessor, CLIPModel
)
import spacy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Loading Models into VRAM...")
nlp = spacy.load("en_core_web_sm")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
Loading Models into VRAM...



## Component A: BLIP (Vision-Language Captioning Model)

<img src="https://miro.medium.com/1*xqZBtczEO7JY3KklrfS-9A.png" width="500" height="auto">

### Role
BLIP is responsible for generating a **semantic natural language description** of the input image.

### Architecture
- Based on a **Vision Transformer (ViT)** encoder and text decoder
- Uses **cross-attention mechanisms** to align image patches with textual tokens
- Performs image-to-text generation using transformer-based decoding

### Function in Pipeline
BLIP converts raw pixel input into a structured semantic representation (caption), which serves as the foundation for downstream NLP and object detection stages.

### Training Background
- Pretrained on large-scale image-text datasets (e.g., COCO, LAION-style datasets)
- Trained using image captioning and image-text matching objectives

### Reference
Li, J., et al. (2022). *BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding.*


In [3]:
# load BLIP model
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic


## Component B: YOLO-World (Open-Vocabulary Object Detector)

<img src="https://images.prismic.io/encord/Ze8K90mNsf2sHf6F_image5.png?auto=format,compress" width="500" height="auto">

### Role
YOLO-World performs **object localization and classification** using dynamically generated text-based class prompts.

### Architecture
- Built on a **CNN-based YOLO detection backbone**
- Performs single-stage object detection (grid-based prediction)
- Extended with **text-conditioned embeddings** for open-vocabulary recognition

### Function in Pipeline
Unlike traditional YOLO models with fixed label sets, YOLO-World accepts **NLP-extracted keywords** as dynamic detection targets, enabling zero-shot object detection.

### Training Background
- Trained on large-scale object detection datasets (e.g., COCO, Objects365)
- Extended with vision-language alignment data for open-vocabulary generalization

### Reference
Cheng, T., et al. (2024). *YOLO-World: Real-Time Open-Vocabulary Object Detection.*


In [4]:
# load YOLO-world
yolo_model = YOLO('yolov8s-world.pt').to(device)


## Component C: CLIP (Contrastive Vision-Language Model)

<img src="https://lh7-rt.googleusercontent.com/docsz/AD_4nXfuLbhTdbh1eMKWSxvQb1_zH_OKoptBXz8IzJC9IHJJvtQGEN58E7N4icTAHnTVRWva0qfWnsAX3jRLxQ2rmajwDdtKgfDvjruOjpKJAfTxX-2Hs2XQ6MceH5YWHwJTWmUN9Mvu0Q?key=QtFNKp6V292AR-QiTSWygw" width="500" height="auto">

### Role
CLIP is responsible for **global scene classification** by measuring semantic similarity between images and text prompts.

### Architecture
- Dual encoder model (Vision Transformer + Text Transformer)
- Learns a **shared embedding space** using contrastive learning
- Computes similarity via cosine/dot-product scoring

### Function in Pipeline
CLIP evaluates the overall scene context by comparing image embeddings against predefined semantic scene descriptions (e.g., indoor, urban, nature).

### Training Background
- Trained on approximately **400 million image-text pairs**
- Uses contrastive loss to align matching image-text representations in a shared latent space

### Reference
Radford, A., et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.*

In [5]:
# load CLIP model
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", torch_dtype=torch.float16).to(device)


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# NLP Parsing for Nouns
This function extracts meaningful keywords (nouns, proper nouns, and pronouns) from a generated caption.
These keywords are then used as inputs for YOLO-World to improve object detection by grounding text into visual entities.

In [6]:
def extract_nouns(caption: str):
    """Extracts critical entities from the BLIP narrative to feed into YOLO-World."""
    doc = nlp(caption.lower())
    ignore_list = ["image", "photo", "picture", "background", "foreground"]

    keywords = [token.text for token in doc
                if token.pos_ in ["NOUN", "PROPN", "PRON"]
                and token.text not in ignore_list]

    return list(set(keywords))



# **End-to-End Application Pipeline**

## **Software Architecture**

The application is built as a **multi-step AI pipeline** that analyze images without needing extra training or fine-tuning. It is split into three main parts:

### **Frontend (Gradio UI)**
- Lets users upload images
- Displays results like detected objects and scene labels
- The visual interface of the system

### **Orchestration Layer (Python + spaCy)**
- Connects all the models together
- Controls the order of operations (what runs first, next, etc.)
- Extracts useful words from captions
- Handles fallback logic if a model fails or returns nothing

### **Model Layer (PyTorch)**
- Runs all deep learning models in **inference mode** (no training happening)
- Includes:
  - **BLIP** → describes the image in words
  - **YOLO-World** → finds objects in the image
  - **CLIP** → understands the overall scene

---

## **How the Models Are Connected**

The models don’t work independently, they pass information to each other:

1. Image is sent to **BLIP**
2. BLIP generates a caption (description of the image)
3. spaCy extracts key words (like nouns)
4. Those words become input for **YOLO**
5. YOLO finds objects based on those words
6. At the same time, **CLIP** analyzes the overall scene

---

## **Neural Network Integration**

All models are run using standard PyTorch inference tools.

### BLIP Integration
- Uses `generate()` to create captions
- Controls output length and quality using parameters like:
  - max length
  - repetition penalty


### YOLO Integration
- Uses `set_classes()` to dynamically decide what to look for
- Runs detection with:
  - confidence threshold (how strict it is)
  - overlap threshold (how boxes are filtered)


### CLIP Integration
- Takes image + text descriptions
- Compares them in a shared space
- Returns similarity scores
- These scores are turned into probabilities using softmax

### **Main Insight**

Instead of using one big AI model, this system combines multiple smaller ones that specialize in different tasks. In addition, The system uses language as a “connector” between models.

This allows:
- No fixed object list
- No retraining required
- Flexible, open-vocabulary understanding of images

In [7]:
@torch.no_grad()
def process_image(image: Image.Image):
    # semantic captioning
    prompt = "a photo of"
    blip_inputs = blip_processor(image, text=prompt, return_tensors="pt").to(device)
    blip_inputs["pixel_values"] = blip_inputs["pixel_values"].to(torch.float16)
    generated_ids = blip_model.generate(
        **blip_inputs,
        max_new_tokens=30,
        num_beams=5,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    caption = blip_processor.decode(generated_ids[0], skip_special_tokens=True).title()

    # object detection
    queries = extract_nouns(caption)
    if not queries:
        queries = ["object"]

    yolo_model.set_classes(queries)
    results = yolo_model.predict(source=image, conf=0.15, iou=0.45, verbose=False)[0]

    raw_detections = []
    for box in results.boxes:
        label_name = yolo_model.names[int(box.cls[0])].title()
        confidence = float(box.conf[0])
        coords = box.xyxy[0].tolist()

        raw_detections.append({
            "label": label_name,
            "confidence": round(confidence * 100, 1),
            "box": [round(i, 2) for i in coords]
        })

    # fall back if did not detect anything
    if len(raw_detections) == 0:
        fallback_classes = ["person", "vehicle", "animal", "furniture", "building", "structure", "bag", "object"]
        yolo_model.set_classes(fallback_classes)

        fallback_results = yolo_model.predict(source=image, conf=0.20, iou=0.45, verbose=False)

        for result in fallback_results:
            for box in result.boxes:
                raw_detections.append({
                    "label": yolo_model.names[int(box.cls[0])].title(),
                    "confidence": round(float(box.conf[0]) * 100, 1),
                    "box": [round(i, 2) for i in box.xyxy[0].tolist()]
                })

    detected_items = sorted(raw_detections, key=lambda x: x["confidence"], reverse=True)[:15]

    # scene classification
    scene_categories = {
        "Indoor Space": "a photo with an indoor room background setting",
        "Nature or Wildlife": "a photo with an outdoor natural background",
        "Urban City": "a photo with an outdoor urban city background",
        "City Skyline": "a photo showing a distant city skyline",
        "Crowded Area": "a photo showing a dense crowd of people"
    }

    ui_labels = list(scene_categories.keys())
    clip_prompts = list(scene_categories.values())

    clip_inputs = clip_processor(
        text=clip_prompts, images=image, return_tensors="pt", padding=True
    ).to(device)

    clip_inputs["pixel_values"] = clip_inputs["pixel_values"].to(torch.float16)
    clip_outputs = clip_model(**clip_inputs)
    probs = clip_outputs.logits_per_image.softmax(dim=1).squeeze().tolist()

    scene_results = sorted(
        [{"label": label, "probability": round(p * 100, 1)} for label, p in zip(ui_labels, probs)],
        key=lambda x: x["probability"],
        reverse=True
    )

    return caption, detected_items, scene_results

# Deployment
To provide a fully functional end-to-end application, the inference engine is wrapped in a **Gradio** UI.

This UI abstracts the complex tensor operations away from the user. It intercepts the raw bounding box coordinates outputted by the backend and utilizes `PIL.ImageDraw` to visually overlay the telemetry onto the source image in real-time.


In [8]:
import gradio as gr
from PIL import ImageDraw, ImageFont
import traceback
import numpy as np
import os
import urllib.request

# download font for our UI
font_path = "Roboto-Bold.ttf"

if not os.path.exists(font_path):
    print("downloading roboto font...")
    urllib.request.urlretrieve(
        "https://github.com/googlefonts/roboto/raw/main/src/hinted/Roboto-Bold.ttf",
        font_path
    )

def inference_wrapper(image):
    if image is None:
        return None, "ERROR: No image detected.", "Waiting for image...", {"Error": 1.0}

    try:
        process_img = image.convert("RGB")

        # main pipeline (blip caption + yolo detections + clip classification)
        caption, detected_items, scene_results = process_image(process_img)

        annotated_image = process_img.copy()
        draw = ImageDraw.Draw(annotated_image)

        # scale font relative to image size
        text_size = max(20, int(annotated_image.width * 0.03))
        font = ImageFont.truetype(font_path, text_size)

        yolo_labels_clean = []

        for item in detected_items:
            box = item["box"]
            label_text = f"{item['label']} ({item['confidence']}%)"
            yolo_labels_clean.append(label_text)

            box_coords = [int(box[0]), int(box[1]), int(box[2]), int(box[3])]

            # scale box thickness with resolution
            line_width = max(3, int(annotated_image.width * 0.006))
            draw.rectangle(box_coords, outline="red", width=line_width)

            text_x = box_coords[0]
            text_y = max(0, box_coords[1] - (text_size + int(text_size * 0.2)))

            # pad background for readability
            text_bbox = draw.textbbox((text_x, text_y), label_text, font=font)
            padding = int(text_size * 0.2)
            padded_bbox = [
                text_bbox[0] - padding,
                text_bbox[1] - padding,
                text_bbox[2] + padding,
                text_bbox[3] + padding
            ]

            draw.rectangle(padded_bbox, fill="red")
            draw.text((text_x, text_y), label_text, fill="white", font=font)

        yolo_text_output = ", ".join(yolo_labels_clean) if yolo_labels_clean else "No identifiable objects detected."

        # normalize to [0,1] for gradio label component
        clip_chart_data = {
            item["label"]: float(item["probability"]) / 100.0
            for item in scene_results
        }

        return np.array(annotated_image), caption, yolo_text_output, clip_chart_data

    except Exception as e:
        print(traceback.format_exc())
        return np.array(image), f"SYSTEM CRASH: {str(e)}", "Error", {"Error": 1.0}

# build UI
with gr.Blocks() as demo:
    gr.Markdown(
        """
        <div style="text-align: center;">
            <h1>Clarity: Semantic Vision Engine</h1>
            <p style="color: gray;">Dynamic Multi-Modal Pipeline: BLIP Narrative ➔ YOLO-World Detection ➔ CLIP Scene Analysis</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column():
            shared_image = gr.Image(type="pil", label="Image Input / Annotated Output", height=550)
            submit_btn = gr.Button("Analyze Image", variant="primary")

        with gr.Column():
            gr.Markdown("<h3 style='text-align: center;'>Scene Intelligence</h3>")
            out_caption = gr.Textbox(label="1. SEMANTIC NARRATIVE (BLIP)", lines=2, interactive=False)
            out_scene = gr.Label(label="2. SCENE CONTEXT (ZERO-SHOT CLIP)", num_top_classes=3)
            out_yolo_text = gr.Textbox(label="3. DETECTED OBJECTS (YOLO-WORLD)", lines=2, interactive=False)

    submit_btn.click(
        fn=inference_wrapper,
        inputs=shared_image,
        outputs=[shared_image, out_caption, out_yolo_text, out_scene]
    )

print("Launching UI...")
demo.launch(debug=True, share=True)

downloading roboto font...
Launching UI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f181416e8bfbacc70.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 36 packages in 711ms
Prepared 2 packages in 2.79s
Installed 2 packages in 2ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@81ff68ed7ffcac3b40484c914f104f816757308d)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 4.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 221MiB/s]


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7f181416e8bfbacc70.gradio.live
